# Setup — Low-Light Enhancement Project

Esegui questo notebook all'inizio di ogni sessione su **Kaggle** per installare le dipendenze e verificare che l'ambiente GPU sia pronto.

**Prima di eseguire:**
- Settings → Accelerator → **GPU T4 x2** (o T4)
- Settings → Internet → **On** (necessario per pip install)

**Limiti Kaggle gratuito:**
- 30 ore/settimana di GPU
- Sessione massima: 12 ore
- Output salvato in `/kaggle/working/` (persiste tra sessioni)

## 1. Installa le dipendenze

In [ ]:
# Su Kaggle torch è già installato — questo aggiornerà solo le librerie mancanti
!pip install -r /kaggle/input/low-light-repo/requirements.txt --quiet

# Alternativa se hai caricato il repo come dataset Kaggle:
# !pip install piq tqdm scikit-image --quiet

## 2. Verifica PyTorch e CUDA

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}           : {props.name}")
        print(f"VRAM totale     : {props.total_memory / 1e9:.1f} GB")
else:
    print("ATTENZIONE: nessuna GPU — abilita T4 in Settings → Accelerator.")

## 3. Verifica Mixed Precision (AMP)

In [ ]:
from torch.cuda.amp import autocast, GradScaler

if torch.cuda.is_available():
    scaler = GradScaler()
    with autocast():
        x = torch.randn(1, 3, 256, 256, device='cuda')
    print(f"AMP OK — dtype tensore: {x.dtype}")
else:
    print("AMP non verificabile senza GPU.")

## 4. Verifica tutte le dipendenze

In [ ]:
import importlib

deps = [
    "torch", "torchvision", "numpy", "PIL",
    "skimage", "piq", "tqdm", "tensorboard", "matplotlib"
]

for dep in deps:
    try:
        mod = importlib.import_module(dep)
        version = getattr(mod, '__version__', 'n/a')
        print(f"  OK       {dep:<15} {version}")
    except ImportError:
        print(f"  MANCANTE {dep}")

## 5. Percorsi Kaggle

In [ ]:
import os

WORKING_DIR   = "/kaggle/working"    # output persistente tra sessioni
INPUT_DIR     = "/kaggle/input"      # dataset aggiunti dal pannello Kaggle
CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints")
RESULTS_DIR   = os.path.join(WORKING_DIR, "results")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Working dir   : {WORKING_DIR}")
print(f"Input dir     : {INPUT_DIR}")
print(f"Checkpoints   : {CHECKPOINT_DIR}")
print(f"Results       : {RESULTS_DIR}")
print("\nDataset disponibili in input:")
for name in os.listdir(INPUT_DIR):
    print(f"  /kaggle/input/{name}/")

## 6. Imposta seed globale

In [ ]:
import sys
sys.path.append("/kaggle/input/low-light-repo")

from src.utils.reproducibility import set_seed, SEED

set_seed(SEED)
print(f"Seed impostato: {SEED}")
print("Ambiente pronto.")